# Sequences

Recurrent models (RNN, LSTM, GRU) process sequences where each time step's
output depends on all previous inputs. In idris-ml, sequence data uses
`RecurrentDataPoint` and training uses `epochRecurrentNativeTensor`.

We'll train an RNN to predict the next element in a repeating [0, 1, 0] pattern.

## Recurrent Data

`RecurrentDataPoint` holds variable-length sequences of fixed-dimension vectors:

In [1]:
:t RecurrentDataPoint

DataPoint.RecurrentDataPoint : Nat -> Nat -> Type -> Type


In [2]:
:t MkRecurrentDataPoint

DataPoint.MkRecurrentDataPoint : List (Vector i ty) -> List (Vector o ty) -> RecurrentDataPoint i o ty


The `.xs` field is a `List (Vector i ty)` — a variable-length sequence of input
vectors, each of fixed dimension `i`. The `.ys` field is the corresponding output
sequence. Different data points can have different sequence lengths.

## Pattern Prediction Task

`patternData` generates sequences of the repeating pattern [0, 1, 0, 0, 1, 0, ...].
The input is the pattern and the target is the next element — the model must learn
the repeating structure.

In [3]:
:t patternData

Generate.patternData : (n : Nat) -> Vect n (RecurrentDataPoint 1 1 Double)


In [4]:
:exec putStrLn (show (xs (index FZ (patternData 4))))

[[0.0], [1.0], [0.0]]


In [5]:
:exec putStrLn (show (ys (index FZ (patternData 4))))

[[1.0], [0.0], [0.0]]


Each data point is a sequence of scalar (1-element) vectors. The target at each
time step is the next element in the pattern.

## RNN Model

An RNN layer maintains hidden state that carries information across time steps.
The `forward` function returns the updated model (with new hidden state) alongside
the output — pure functional, no mutation.

In [6]:
:t rnnLayer

Layer.Rnn.rnnLayer : (Num ty, FromDouble ty) => IO (AnyLayer i o ty)


In [7]:
:exec do { srand 42;
  rnn <- rnnLayer {i=1, o=1};
  model <- pure (autoName (OutputLayer rnn));
  putStrLn ("Model: " ++ show model) }

Error: Can't find an implementation for Show (Network 1 [] 1 (Variable ?d)).

(Interactive):1:117--1:127
 1 | :exec do { srand 42; rnn <- rnnLayer {i=1, o=1}; model <- pure (autoName (OutputLayer rnn)); putStrLn ("Model: " ++ show model) }
                                                                                                                         ^^^^^^^^^^


## Recurrent Forward Pass

`forwardRecurrent` processes a full sequence, threading hidden state through
each time step. It returns a list of outputs, one per time step.

In [8]:
:t forwardRecurrent

Layer.Core.forwardRecurrent : (FromDouble ty, (Floating ty, (Fractional ty, (Neg ty, (Num ty, Ord ty))))) => Network i hs o ty -> List (Vector i ty) -> (Network i hs o ty, List (Vector o ty))


## Training

Recurrent training uses `epochRecurrentNativeTensor` and binary cross-entropy loss
(the output is a probability of the next element being 1).

In [9]:
:exec do { srand 42;
  rnn <- rnnLayer {i=1, o=1};
  model <- pure (autoName (OutputLayer rnn));
  opt <- pure (nativeSgd 0.03);
  (trained, epochs, loss) <- runTraining
    (\m, d => epochRecurrentNativeTensor opt d bceTensor m)
    (pure (patternData 8)) (simpleConfig 500) model;
  putStrLn ("Trained " ++ show epochs ++ " epochs, loss=" ++ show loss) }

Training... [backend=tape]
  [00:00:00] 0	loss=0.6768558815754979
  [00:00:01] 100	loss=0.6294778493817164
  [00:00:02] 200	loss=0.5692192819687022
  [00:00:03] 300	loss=0.5278484685289803
  [00:00:04] 400	loss=0.498151133334594
Completed in 5s (500 epochs, 10ms/epoch)
Trained 500 epochs, loss=0.47289757767085877


## Evaluation

After training, check if the model predicts the pattern correctly.
Convert to Double for evaluation, then run `evaluateRecurrent`.

In [10]:
:exec do { srand 42;
  rnn <- rnnLayer {i=1, o=1};
  model <- pure (autoName (OutputLayer rnn));
  opt <- pure (nativeSgd 0.03);
  (trained, epochs, loss) <- runTraining
    (\m, d => epochRecurrentNativeTensor opt d bceTensor m)
    (pure (patternData 8)) (simpleConfig 1000) model;
  dblModel <- pure (toDoubleNetwork (emap refreshValue trained));
  preds <- pure (evaluateRecurrent dblModel (patternData 4));
  putStrLn ("Loss: " ++ show loss);
  putStrLn "";
  putStrLn "Pattern: [0,1,0,0,1,0,...] -> predict next";
  traverse_ (\p =>
    putStrLn ("  predicted: " ++ show (map (map (\x => if x > 0 then 1 else 0)) p)))
    (toList preds);
  putStrLn "";
  putStrLn "(Values > 0 map to 1, <= 0 map to 0)" }

Training... [backend=tape]
  [00:00:00] 0	loss=0.6768558815754979
  [00:00:01] 100	loss=0.6294778493817164
  [00:00:01] 200	loss=0.5692192819687022
  [00:00:02] 300	loss=0.5278484685289803
  [00:00:03] 400	loss=0.498151133334594
  [00:00:04] 500	loss=0.4726587768794381
  [00:00:05] 600	loss=0.450187707381949
  [00:00:06] 700	loss=0.43015479817829727
  [00:00:07] 800	loss=0.41214523089330823
  [00:00:08] 900	loss=0.3958459323258824
Completed in 9s (1000 epochs, 9ms/epoch)
Loss: 0.38115285802183874

Pattern: [0,1,0,0,1,0,...] -> predict next
  predicted: [[1], [0], [0]]
  predicted: [[1], [0], [0], [1]]
  predicted: [[1], [0], [0], [1], [0]]
  predicted: [[1], [0], [0], [1], [0], [0]]

(Values > 0 map to 1, <= 0 map to 0)


## LSTM: Drop-In Replacement

LSTM is a more powerful recurrent layer that handles longer dependencies.
The API is identical — just swap `rnnLayer` for `lstmLayer`:

In [11]:
:t lstmLayer

Layer.Lstm.lstmLayer : (Num ty, FromDouble ty) => IO (AnyLayer i o ty)


In [12]:
:exec do { srand 42;
  lstm <- lstmLayer {i=1, o=1};
  model <- pure (autoName (OutputLayer lstm));
  opt <- pure (nativeSgd 0.03);
  (trained, epochs, loss) <- runTraining
    (\m, d => epochRecurrentNativeTensor opt d bceTensor m)
    (pure (patternData 8)) (simpleConfig 500) model;
  putStrLn ("LSTM - " ++ show epochs ++ " epochs, loss=" ++ show loss) }

Training... [backend=tape]
  [00:00:00] 0	loss=0.6814106634356146
  [00:00:01] 100	loss=0.6782002847888179
  [00:00:02] 200	loss=0.6752832266676235
  [00:00:03] 300	loss=0.6724802468800261
  [00:00:04] 400	loss=0.6696455328692483
Completed in 5s (500 epochs, 10ms/epoch)
LSTM - 500 epochs, loss=0.6667065626383866


Same data, same optimizer, same training loop — just a different layer.
The type system ensures the dimensions still match.

## When to Use What

| Model | Best for | Trade-off |
|-------|----------|-----------|
| RNN | Short sequences, simple patterns | Fast but forgets long-range dependencies |
| LSTM | Medium sequences, complex patterns | More parameters, better memory |
| GRU | Similar to LSTM, fewer parameters | Simpler gating, often comparable accuracy |
| Transformer | Long sequences, parallel processing | `make example-transformer` for a demo |

For longer examples with more epochs:
```bash
make example-rnn      # RNN on pattern task (2000 epochs)
make example-lstm     # LSTM on same task with early stopping
```